# Assignment: Extend the az.ipynb Lab

**Based on:** `Lab2.ipynb` (the Module 3 lab).

This week's assignment is short on purpose: take your working `Lab2.ipynb` lab and add **one
more step** to the chain. No new concepts, no new setup, no new libraries — just one more
chained LLM call that builds on what you already have.

**Two deployments this time:** `gpt-5.1-ptu` is the default deployment for every existing
step (Steps 1–4). The new step you add (Step 5) must call `gpt-5.4-ptu` instead.

## Step 1 — Start from your working lab

- Make a copy of your completed `Lab2.ipynb` (e.g. rename the copy `assignment3.ipynb`), or
  continue directly inside this notebook — either is fine.
- Copy in your working code from the lab's Steps 1–4: the imports and `.env` config, the
  `AzureOpenAI` client, the `chat()` helper, and the chain itself (fun fact → generate a hard
  question → answer it → evaluate the answer).
- Confirm your `.env`'s `AZURE_APIM_OPENAI_DEPLOYMENT` is set to `gpt-5.1-ptu` — this stays
  the default deployment for Steps 1–4, unchanged.

Run those cells first and confirm they still work before moving on.

## 1. Imports

- `dotenv.load_dotenv` reads key/value pairs from a local `.env` file into environment variables.
- `AzureOpenAI` is the Azure-flavored client class from the `openai` package (as opposed to the
  plain `OpenAI` class used for api.openai.com).

In [21]:
from dotenv import load_dotenv
import os
import sys

from openai import AzureOpenAI

## 2. Load environment variables

`load_dotenv(override=True)` looks for a `.env` file in the current directory and loads its
contents into `os.environ`. `override=True` means values in `.env` take priority over any values
already set in the shell environment.

Make sure you have a `.env` file (in the same folder as this notebook, or on `sys.path`) that
defines:

```
AZURE_APIM_OPENAI_SUBSCRIPTION_KEY=...
AZURE_APIM_OPENAI_API_VERSION=...
AZURE_APIM_OPENAI_ENDPOINT=https://<your-apim-instance>.azure-api.net
AZURE_APIM_OPENAI_DEPLOYMENT=<your-deployment-name>
```

**Note on the endpoint:** it should be the *host only* — e.g.
`https://apim-azr-ue2-bgpt-prd-ucin.azure-api.net` — **without** a trailing
`/openai/deployments/...` path. The SDK builds the full path itself using the API version and
deployment name.

In [22]:
# Read .env and override any existing process env values.
load_dotenv(override=True)

# APIM settings from .env. Endpoint must be the host only, e.g.
# https://apim-azr-ue2-bgpt-prd-ucin.azure-api.net  (no /openai/deployments)
api_key = os.getenv("AZURE_APIM_OPENAI_SUBSCRIPTION_KEY")
api_version = os.getenv("AZURE_APIM_OPENAI_API_VERSION")
endpoint = os.getenv("AZURE_APIM_OPENAI_ENDPOINT")
deployment = os.getenv("AZURE_APIM_OPENAI_DEPLOYMENT")

## 3. Validate configuration

Before making any network calls, confirm that all four required settings were actually found.
`all([...])` returns `False` if any of them is `None` or an empty string, in which case
`sys.exit(...)` prints a helpful message and stops execution (raises `SystemExit` — in a notebook
this will show as an error, which is expected if `.env` is missing or incomplete).

If configuration is present, we print a masked preview of the API key (first 8 characters only)
and the deployment name, just to confirm — without ever logging the key can be verified without
leaking the whole secret.

In [23]:
if not all([api_key, api_version, endpoint, deployment]):
    sys.exit(
        "Missing Azure APIM settings. Set AZURE_APIM_OPENAI_SUBSCRIPTION_KEY, "
        "AZURE_APIM_OPENAI_API_VERSION, AZURE_APIM_OPENAI_ENDPOINT, and "
        "AZURE_APIM_OPENAI_DEPLOYMENT in your .env file."
    )

print(f"Azure APIM key exists and begins {api_key[:8]}")
print(f"Deployment: {deployment}")

Azure APIM key exists and begins b1e65bd7
Deployment: gpt-5.1-ptu


## 4. Create the Azure OpenAI client

`AzureOpenAI` is instantiated once and reused for every request. Note the three arguments it
needs, which differ from the plain `OpenAI` client:

- `api_key` — here it's actually the APIM **subscription key**, not an Azure OpenAI resource key,
  since requests are routed through the APIM gateway.
- `api_version` — the Azure OpenAI REST API version (e.g. `2024-06-01`), required because Azure
  versions its API explicitly, unlike api.openai.com.
- `azure_endpoint` — the base host of the APIM instance (see the note above).

In [24]:
# Sync client pointed at Azure APIM.
client = AzureOpenAI(
    api_key=api_key,
    api_version=api_version,
    azure_endpoint=endpoint,
)

## 5. A small `chat()` helper

This wraps `client.chat.completions.create(...)` so the rest of the notebook doesn't repeat
boilerplate. Two Azure-specific details are worth calling out:

- **`model=deployment`** — On Azure, the `model` parameter is actually the *deployment name* you
  configured in the Azure OpenAI resource (e.g. `gpt-5-chat`), not a generic model id like
  `gpt-5`. Azure routes the request based on that deployment.
- **`max_completion_tokens` instead of `max_tokens`** — Newer reasoning-capable deployments
  (GPT-5-style) reject the older `max_tokens` parameter and require `max_completion_tokens`
  instead. The helper defaults this to `5000` but lets callers override it.

An optional `deployment` argument defaults to the module-level `deployment` from `.env`, so a
later step can override it (e.g. `chat(messages, deployment="gpt-5.4-ptu")`).

In [25]:
# On Azure the `model` argument is the *deployment name*, not an OpenAI model id.
# GPT-5 deployments need max_completion_tokens (max_tokens is rejected).
def chat(messages, max_completion_tokens=5000, deployment=deployment):
    return client.chat.completions.create(
        model=deployment,
        messages=messages,
        max_completion_tokens=max_completion_tokens,
    )

## 6. Step 1 — Ask for a fun fact

The simplest possible call: a single user message, default token limit, and we print the
model's reply text (`response.choices[0].message.content`).

In [26]:
# 1) Fun fact
messages = [{"role": "user", "content": "Tell me a short fun fact"}]
response = chat(messages)
fact = response.choices[0].message.content
print(fact)

Octopuses have three hearts: two pump blood to the gills, and one pumps it to the rest of the body. When they swim, the main heart actually stops beating, which is one reason they prefer crawling to swimming.


## 7. Step 2 — Ask the model to invent a hard question

We prompt the model to generate a challenging, IQ-style question, instructing it to respond
**only** with the question itself (no preamble) so that `question` can be reused directly as the
next prompt.

In [27]:
# 2) Ask the model to invent a hard IQ-style question
question = (
    "Please propose a hard, challenging question to assess someone's IQ. "
    "Respond only with the question."
)
messages = [{"role": "user", "content": question}]
response = chat(messages)
question = response.choices[0].message.content
print(question)

You have 12 visually identical coins and a balance scale. Exactly one coin is counterfeit; it differs in weight from the genuine coins, but you don’t know whether it is heavier or lighter. Using the balance scale at most three times, determine which coin is counterfeit and whether it is heavier or lighter. Describe a complete strategy that always works, no matter which coin is counterfeit or whether it is heavier or lighter.


## 8. Step 3 — Ask the model to answer its own question

The `question` text generated in the previous step is now sent back to the model as a fresh
prompt (a brand-new `messages` list — the model has no memory of generating the question; it's
just answering it as if seeing it cold).

Since we're already in a notebook, we can render the answer as nicely formatted Markdown using
`IPython.display` directly — no fallback needed.

In [28]:
# 3) Ask the model to answer that question
messages = [{"role": "user", "content": question}]
response = chat(messages)
answer = response.choices[0].message.content
print(answer)

Label the coins 1–12. We’ll use three weighings. On each weighing we’ll note not just which side is heavier or lighter, but the *pattern* of results across all three weighings to identify the coin and whether it is heavy or light.

Notation:  
- “L” = left side heavier  
- “R” = right side heavier  
- “B” = balance (equal)

---

## Weighing plan

### Weighing 1
Weigh:  
**1, 2, 3, 4** (left) vs **5, 6, 7, 8** (right)

- If it balances (B): the counterfeit is among {9,10,11,12}.  
- If it doesn’t balance: the counterfeit is among {1,2,3,4,5,6,7,8}, and we know which side heavier for later deduction.

We divide into two main branches:

---

## Case A: First weighing balances (counterfeit in 9–12)

So W1: **9–12 are suspects**, 1–8 genuine.

### Weighing 2 (Case A)
Weigh:  
**1, 2, 9, 10** (left) vs **3, 4, 11, 12** (right)

- If this balances too (B): then 9–12 are all genuine, impossible. So must be unbalanced (L or R).

Let’s analyze:

1. **W1: B, W2: L (left heavier)**  
   Possibilit

In [29]:
# Render the answer as Markdown in the notebook.
from IPython.display import Markdown, display

display(Markdown(answer))

Label the coins 1–12. We’ll use three weighings. On each weighing we’ll note not just which side is heavier or lighter, but the *pattern* of results across all three weighings to identify the coin and whether it is heavy or light.

Notation:  
- “L” = left side heavier  
- “R” = right side heavier  
- “B” = balance (equal)

---

## Weighing plan

### Weighing 1
Weigh:  
**1, 2, 3, 4** (left) vs **5, 6, 7, 8** (right)

- If it balances (B): the counterfeit is among {9,10,11,12}.  
- If it doesn’t balance: the counterfeit is among {1,2,3,4,5,6,7,8}, and we know which side heavier for later deduction.

We divide into two main branches:

---

## Case A: First weighing balances (counterfeit in 9–12)

So W1: **9–12 are suspects**, 1–8 genuine.

### Weighing 2 (Case A)
Weigh:  
**1, 2, 9, 10** (left) vs **3, 4, 11, 12** (right)

- If this balances too (B): then 9–12 are all genuine, impossible. So must be unbalanced (L or R).

Let’s analyze:

1. **W1: B, W2: L (left heavier)**  
   Possibilities:
   - 9 or 10 could be **heavy** (on heavier left)
   - 11 or 12 could be **light** (on lighter right)

   ### Weighing 3 (subcase A1)
   Weigh: **9** vs **10**
   - If 9 > 10: coin 9 is **heavy**.
   - If 9 < 10: coin 10 is **heavy**.
   - If 9 = 10: both are normal → the counterfeit is either 11 or 12 and is **light**.  
     But we must distinguish 11 vs 12:
       - Compare **11** vs a known good coin (e.g. 1):
         - If 11 is lighter: 11 is light.
         - If 11 balances: 12 is light.

2. **W1: B, W2: R (right heavier)** (mirror situation)  
   Possibilities:
   - 11 or 12 could be **heavy**
   - 9 or 10 could be **light**

   ### Weighing 3 (subcase A2)
   Weigh: **11** vs **12**
   - If 11 > 12: 11 is **heavy**.
   - If 11 < 12: 12 is **heavy**.
   - If 11 = 12: both normal → counterfeit is 9 or 10, and is **light**.  
     Then weigh **9** vs a known good coin:
     - If 9 is lighter: 9 is light.
     - If 9 balances: 10 is light.

This completes the case where W1 balances.

---

## Case B: First weighing does not balance (counterfeit in 1–8)

Assume, without loss of generality, that in W1 the left side is heavier:  
W1: **1,2,3,4 (L)** vs **5,6,7,8 (R)**, and result is L (left heavier).

Interpretation:
- A heavy counterfeit could be among {1,2,3,4}.  
- A light counterfeit could be among {5,6,7,8}.

### Weighing 2 (Case B)
Weigh:  
**1, 2, 5** (left) vs **3, 6, 9** (right)

(9 is known genuine if counterfeit is in 1–8, but we don’t yet know that in logic; the pattern of results will enforce consistency.)

Now examine combinations of W1 and W2:

---

### Subcase B1: W1 = L, W2 = L (left heavier again)

From W1 and W2:

- If coin 1 were heavy:
  - W1: heavier left (fits, 1 on left)  
  - W2: heavier left (fits, 1 on left)
- If coin 2 were heavy: similarly fits.
- If coin 3 were heavy:
  - W1: heavier left (fits, 3 on left)
  - W2: heavier right (would be right heavier, since 3 on the right) ⇒ contradicts W2=L.  
  So 3 cannot be heavy.
- If coin 4 were heavy: appears only in W1; that still works (contributes to left being heavy in W1, neutral in W2).
- If coin 5 were light:
  - W1: right lighter makes left heavier (fits)
  - W2: coin 5 on left, so left would be lighter, contradicting W2=L. So 5 cannot be light.
- If coin 6 were light:
  - W1: 6 on right, making right lighter → left heavier (fits)
  - W2: 6 on right, making right lighter → right lighter, so left heavier (fits).
- If coin 7 or 8 were light: only affect W1 and are consistent.

So possible suspects after “L, L” are:  
- Heavy: 1, 2, 4  
- Light: 6, 7, 8  

### Weighing 3 (Subcase B1)
Weigh: **1** vs **2**

- If 1 > 2: coin 1 is **heavy**.  
- If 1 < 2: coin 2 is **heavy**.  
- If 1 = 2: both normal → heavy candidate 4 is still possible, and light candidates 6,7,8 remain.

We must separate 4 (heavy) from {6,7,8} (light).

Now weigh: **4** vs a known normal coin (e.g. 9):
- If 4 is heavier: 4 is **heavy**.
- If 4 balances: counterfeit is one of 6,7,8 and **light**.

Finally, distinguish among 6,7,8 (if needed) by an additional comparison with a known normal coin.  
(Here we see that trying to do it linearly like this takes more than 3 weighings; so we instead use the known complete scheme given below.)

---

At this point, it’s clearer to switch to a standard, symmetric complete scheme for all subcases. The fully worked, correct strategy is:

---

## Standard Symmetric Strategy (Complete and Correct)

Label coins A,B,C,D,E,F,G,H,J,K,L,M (12 coins). We’ll describe weighings and then decode from patterns.

### Weighing 1
A, B, C, D vs E, F, G, H

- If they balance, counterfeit is in {J,K,L,M}.  
- If left heavy, counterfeit is among {A,B,C,D,E,F,G,H}; either A–D heavy or E–H light.  
- If right heavy, symmetric.

Define outcome symbols:
- W1, W2, W3 each ∈ {L,R,B}.  
We will assign coins so that every coin/heavy-or-light case has a unique (W1,W2,W3) pattern.

The full, known solution (one of many valid ones) is:

### Weighing assignments

1. **Weighing 1:**  
   Left: A B C D  
   Right: E F G H  

2. **Weighing 2:**  
   Left: A B E J  
   Right: C F K L  

3. **Weighing 3:**  
   Left: A C F M  
   Right: B D G J  

Now, from the three outcomes (e.g. LRB), we can identify exactly which coin is counterfeit and whether it is heavy or light using a precomputed table. Each possibility (12 coins × {heavy,light} = 24 cases) maps to a distinct triple of outcomes from these weighings.

Because describing the entire 24-row decoding table in text is long and mechanical, the essential content of the strategy is:

- The 3 weighings above are fixed in advance (they do not depend on earlier results).
- For each possible counterfeit coin and whether it is heavy or light, the resulting pattern (W1, W2, W3) is unique.
- Therefore, after at most three weighings, you read off the pattern and determine both:
  - which coin is counterfeit, and  
  - whether it is heavier or lighter.

This is known to be optimal: 3 weighings give at most 3³ = 27 distinct outcome patterns; we need to distinguish 24 possibilities (12 coins × 2 types), so a 3-weighing scheme can exist, and the one above is such a complete scheme.

## 9. Step 4 — Ask the model to evaluate the answer

Finally, we build a single prompt string that includes both the `question` and the `answer`
(using an f-string), and ask the model to judge whether the answer is correct. This is a common
"LLM-as-judge" pattern: use the same (or another) model to self-critique its own prior output.

This call uses a higher `max_completion_tokens` (5000) since an evaluation with reasoning tends
to need more room than a short fun fact. Steps 1–4 all use the default deployment from `.env`.

In [30]:
# 4) Ask the model to evaluate the answer
message = f"""
Here is a question:
{question}

And here is a possible answer that might be correct or incorrect:
{answer}

Please evaluate if the answer is correct or incorrect.
"""
print(message)


Here is a question:
You have 12 visually identical coins and a balance scale. Exactly one coin is counterfeit; it differs in weight from the genuine coins, but you don’t know whether it is heavier or lighter. Using the balance scale at most three times, determine which coin is counterfeit and whether it is heavier or lighter. Describe a complete strategy that always works, no matter which coin is counterfeit or whether it is heavier or lighter.

And here is a possible answer that might be correct or incorrect:
Label the coins 1–12. We’ll use three weighings. On each weighing we’ll note not just which side is heavier or lighter, but the *pattern* of results across all three weighings to identify the coin and whether it is heavy or light.

Notation:  
- “L” = left side heavier  
- “R” = right side heavier  
- “B” = balance (equal)

---

## Weighing plan

### Weighing 1
Weigh:  
**1, 2, 3, 4** (left) vs **5, 6, 7, 8** (right)

- If it balances (B): the counterfeit is among {9,10,11,12}. 

In [31]:
messages = [{"role": "user", "content": message}]
response = chat(messages, max_completion_tokens=5000)
evaluation = response.choices[0].message.content
print(evaluation)

The proposed answer is **not fully correct** for this question as stated.

Here’s why:

1. **Incomplete in Case A (W1 balances)**  
   The answer’s handling of the “first weighing balances” branch (counterfeit in coins 9–12) is logically fine *up to a point*, but it requires as many as **four weighings** in some subcases, even though the problem only allows **three total weighings**.

   Specifically:
   - W1: balance → suspect {9,10,11,12}.
   - W2: `1,2,9,10` vs `3,4,11,12`.
   - Suppose W2 = L (left heavier).  
     Then:
     - W3: 9 vs 10.  
       If 9 ≠ 10, you’re done (heavy coin found).  
       If 9 = 10, they propose yet another weighing (11 vs 1) to distinguish between 11 and 12 being light.  
       That is a **fourth** weighing, exceeding the limit.

   The same issue arises symmetrically when W2 = R. So the branch “Case A” is not a valid 3-weighing strategy.

2. **Case B is explicitly abandoned midstream**  
   In the branch where the first weighing does not balance (cou

## Step 2 — Add one more chained step

Add a **5th step** to the chain. Its prompt must be built from at least one variable you
already have (`fact`, `question`, `answer`, or the evaluation text) — the same chaining
pattern as every other step in the lab.

**This step must call `gpt-5.4-ptu`, not the default deployment.** Pass it explicitly when
you call `chat()`:

```python
response = chat(messages, deployment="gpt-5.4-ptu")
```

Pick **one** idea below, or invent your own:

- Rate the difficulty of the question on a 1–10 scale, with a one-sentence justification.
- Rewrite the answer in one simple sentence a 10-year-old could understand.
- Suggest one new, related fun fact that connects to the original topic.
- Translate the final answer into a language of your choice.
- Write a one-line verdict on whether the model's own answer was actually correct, and why.

Store the result in its own variable, and print it clearly labeled (e.g. `=== STEP 5
(gpt-5.4-ptu) ===`).

In [32]:
# TODO: Step 5 - build a new prompt using an earlier variable
#   (fact, question, answer, and/or the evaluation)
message = f"""Given the fact {fact} 
Can you explain it in a way that a 10-year-old can understand?
"""
# TODO: call chat(messages, deployment="gpt-5.4-ptu") -- do NOT use the default deployment here
messages = [{"role": "user", "content": message}]
response = chat(messages, deployment="gpt-5.4-ptu")
res = response.choices[0].message.content
# TODO: extract the result, and print it clearly labeled
display(Markdown(f"=== STEP 5(gpt-5.4-ptu) === <br> {res}"))
#print(res)

=== STEP 5(gpt-5.4-ptu) === <br> Sure! Here’s a kid-friendly way to think about it:

An octopus has **3 hearts**.

- **2 hearts** are like helpers that send blood to the **gills** so the octopus can get oxygen from the water.
- The **1 big heart** sends the oxygen-rich blood to the rest of the body, like the arms and brain.

But here’s the weird part:

When an octopus **swims fast**, the big heart can **stop beating for a little while**. That makes swimming really tiring for the octopus.

So instead of swimming a lot, octopuses usually like to **crawl along the ocean floor**. Crawling uses less energy, so it’s easier for them.

You can imagine it like this:

- The octopus has **three pumps**
- Two pumps fill the blood with oxygen
- One pump sends it everywhere else
- But when it swims, the main pump has trouble keeping up

That’s why octopuses often choose walking/crawling over swimming.

If you want, I can also turn this into a **super short version** or a **fun cartoon-style explanation**.

## Reflection

Answer in a sentence or two each:

1. **Which earlier variable(s) did your Step 5 prompt use, and why that one?**
2. **What would break if you ran Step 5 before the step it depends on?**
3. **Why might a real project deliberately use a different deployment (e.g. a stronger or
   more expensive model) for just one step in a chain, instead of using it everywhere?**

### My reflection

1. I used the *fact* variable because it was giving an interesting fact that I just knew about it. I think it also worked really well if I ask the follow up question of giving some other facts related to the octopus.
2. First, the model may not be called because the chat function and model call hasn't been run. Second, it would break if the answer *fact* from the previous block isn't run, I can't get that value *fact* to plug in the model and ask the follow up question.
3. Depending on different tasks, we may need to consider using different model because different models bring different computing power and price. With difficult and complex tasks, we may need stronger model to get the work done efficiently and get the most value out of it. For simple tasks, we may need light model because simple tasks doesn't require much computing power and it costs less token consumption, deploying light model will be enough to help us with simple tasks. If we plug strong and expensive models, we may overengineer simple tasks.

## Submission checklist

- [X] Notebook runs top to bottom without errors (`Kernel → Restart & Run All`)
- [X] `.env` file is **not** included in your submission
- [X] Step 5 is clearly labeled and its prompt uses at least one earlier variable
- [X] Reflection questions are answered